In [3]:

import pandas as pd
import numpy as np
from IPython.display import display

# ==================================================
# 1. LOAD DATASET
# ==================================================
df = pd.read_csv(
    "HACKATHON_TRAINING_DATA.CSV",
    low_memory=False
)

print("=" * 60)
print("1. DATASET OVERVIEW")
print("=" * 60)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("Total cells:", df.size)
print("Duplicate rows:", df.duplicated().sum())
print("Memory usage (MB):", round(df.memory_usage(deep=True).sum() / 1e6, 2))


# ==================================================
# 2. ALL COLUMN NAMES AND DATA TYPES
# ==================================================
print("\n" + "=" * 60)
print("2. COLUMN NAMES AND DATA TYPES")
print("=" * 60)

column_info = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Unique_Values": df.nunique(dropna=True).values,
    "Missing_Count": df.isnull().sum().values,
    "Missing_Percentage": (
        df.isnull().mean().values * 100
    ).round(2)
})

display(column_info)

print("\nAll column names:")
print(df.columns.tolist())


# ==================================================
# 3. MISSING VALUES
# ==================================================
print("\n" + "=" * 60)
print("3. MISSING VALUE ANALYSIS")
print("=" * 60)

missing = df.isnull().sum()
missing_report = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": (missing / len(df) * 100).round(2)
}).sort_values("Missing_Count", ascending=False)

print("Total missing cells:", int(missing.sum()))
print("Columns with missing values:", int((missing > 0).sum()))

display(missing_report[missing_report["Missing_Count"] > 0])


# ==================================================
# 4. TARGET COLUMN ANALYSIS
# ==================================================
print("\n" + "=" * 60)
print("4. TARGET ANALYSIS")
print("=" * 60)

target = "TARGET"

if target in df.columns:
    print("Target missing values:", df[target].isnull().sum())
    print("\nTarget counts:")
    display(df[target].value_counts(dropna=False).to_frame("Count"))

    print("\nTarget percentages:")
    display(
        (df[target].value_counts(normalize=True, dropna=False) * 100)
        .round(2).to_frame("Percentage")
    )
else:
    print("TARGET column not found. Check the column names.")


# ==================================================
# 5. NUMERICAL AND CATEGORICAL COLUMNS
# ==================================================
print("\n" + "=" * 60)
print("5. NUMERICAL VS CATEGORICAL COLUMNS")
print("=" * 60)

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical columns:", len(numeric_cols))
print(numeric_cols)

print("\nCategorical columns:", len(categorical_cols))
print(categorical_cols)


# ==================================================
# 6. NUMERICAL SUMMARY STATISTICS
# ==================================================
print("\n" + "=" * 60)
print("6. NUMERICAL SUMMARY")
print("=" * 60)

display(df[numeric_cols].describe().T)


# ==================================================
# 7. CATEGORICAL VALUE DISTRIBUTIONS
# ==================================================
print("\n" + "=" * 60)
print("7. CATEGORICAL VALUE DISTRIBUTIONS")
print("=" * 60)

for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(15))


# ==================================================
# 8. CONSTANT AND HIGH-CARDINALITY COLUMNS
# ==================================================
print("\n" + "=" * 60)
print("8. CONSTANT / HIGH-CARDINALITY COLUMNS")
print("=" * 60)

constant_cols = [
    col for col in df.columns
    if df[col].nunique(dropna=False) <= 1
]

print("Constant columns:", constant_cols)

high_cardinality = pd.DataFrame({
    "Unique_Count": df.nunique(dropna=True),
    "Unique_Percentage": (
        df.nunique(dropna=True) / len(df) * 100
    ).round(2)
}).sort_values("Unique_Count", ascending=False)

print("\nColumns with the highest number of unique values:")
display(high_cardinality.head(20))


# ==================================================
# 9. POSSIBLE ID COLUMNS
# ==================================================
print("\n" + "=" * 60)
print("9. IDENTIFIER CHECK")
print("=" * 60)

for col in ["UNIQUE_ID", "TARGET", "TIME_PERIOD"]:
    if col in df.columns:
        print(
            col,
            "| Unique:", df[col].nunique(dropna=True),
            "| Missing:", df[col].isnull().sum()
        )


# ==================================================
# 10. POTENTIAL NUMERICAL OUTLIERS
# ==================================================
print("\n" + "=" * 60)
print("10. POTENTIAL OUTLIERS (IQR METHOD)")
print("=" * 60)

outlier_rows = []

for col in numeric_cols:
    if col == target:
        continue

    series = df[col].dropna()

    if len(series) == 0 or series.nunique() <= 2:
        continue

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    if iqr == 0:
        continue

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((series < lower) | (series > upper)).sum()

    outlier_rows.append({
        "Column": col,
        "Lower_Bound": round(lower, 2),
        "Upper_Bound": round(upper, 2),
        "Outlier_Count": int(count),
        "Outlier_Percentage": round(count / len(series) * 100, 2)
    })

outlier_report = pd.DataFrame(outlier_rows)

if not outlier_report.empty:
    display(
        outlier_report.sort_values(
            "Outlier_Percentage", ascending=False
        ).head(20)
    )


# ==================================================
# 11. CORRELATION WITH TARGET
# ==================================================
print("\n" + "=" * 60)
print("11. NUMERICAL CORRELATION WITH TARGET")
print("=" * 60)

if target in df.columns and pd.api.types.is_numeric_dtype(df[target]):
    correlations = (
        df[numeric_cols]
        .corr(numeric_only=True)[target]
        .drop(target, errors="ignore")
        .dropna()
        .sort_values(key=abs, ascending=False)
    )

    display(correlations.head(20).to_frame("Correlation_with_TARGET"))

print("\nEDA COMPLETE")


1. DATASET OVERVIEW
Number of rows: 327741
Number of columns: 139
Total cells: 45555999
Duplicate rows: 0
Memory usage (MB): 576.1

2. COLUMN NAMES AND DATA TYPES


,Column,Data_Type,Unique_Values,Missing_Count,Missing_Percentage
0,ACCT_AGE,float64,7534,0,0.00
1,LIMIT,float64,18196,0,0.00
2,OUTS,float64,320626,0,0.00
3,ACCT_RESIDUAL_TENURE,float64,12293,13,0.00
4,LOAN_TENURE,int64,1803,0,0.00
...,...,...,...,...,...
134,LATEST_CR_DAYS,float64,1323,1464,0.45
135,LATEST_DR_DAYS,int64,3,0,0.00
136,TIME_PERIOD,object,3,0,0.00
137,TARGET,int64,2,0,0.00



All column names:
['ACCT_AGE', 'LIMIT', 'OUTS', 'ACCT_RESIDUAL_TENURE', 'LOAN_TENURE', 'INSTALAMT', 'SI_FLG', 'AGE', 'VINTAGE', 'KYC_SCR', 'LOCKER_HLDR_IND', 'UID_FLG', 'KYC_FLG', 'INB_FLG', 'EKYC_FLG', 'ONEMNTHCR', 'ONEMNTHSDR', 'ONEMNTHOUTSTANGBAL', 'ONEMNTHAVGMTD', 'ONEMNTHAVGQTD', 'ONEMNTHAVGYTD', 'TWOMNTHSCR', 'TWOMNTHSDR', 'TWOMNTHOUTSTANGBAL', 'TWOMNTHAVGMTD', 'TWOMNTHAVGQTD', 'TWOMNTHAVGYTD', 'THREEMNTHSCR', 'THREEMNTHSDR', 'THREEMNTHOUTSTANGBAL', 'THREEMNTHAVGMTD', 'THREEMNTHAVGQTD', 'THREEMNTHAVGYTD', 'FOURMNTHSCR', 'FOURMNTHSDR', 'FOURMNTHOUTSTANGBAL', 'FOURMNTHAVGMTD', 'FOURMNTHAVGQTD', 'FOURMNTHAVGYTD', 'FIVEMNTHSCR', 'FIVEMNTHSDR', 'FIVEMNTHOUTSTANGBAL', 'FIVEMNTHAVGMTD', 'FIVEMNTHAVGQTD', 'FIVEMNTHAVGYTD', 'SIXMNTHSCR', 'SIXMNTHSDR', 'SIXMNTHOUTSTANGBAL', 'SIXMNTHAVGMTD', 'SIXMNTHAVGQTD', 'SIXMNTHAVGYTD', 'SEVENMNTHSCR', 'SEVENMNTHSDR', 'SEVENMNTHOUTSTANGBAL', 'SEVENMNTHAVGMTD', 'SEVENMNTHAVGQTD', 'SEVENMNTHAVGYTD', 'EIGHTMNTHSCR', 'EIGHTMNTHSDR', 'EIGHTMNTHOUTSTANGBAL'

,Missing_Count,Missing_Percentage
LAST_1_YR_RG4,303363,92.56
LAST_3_YR_RG4,267761,81.70
LATEST_NPA_TENURE,215669,65.80
NO_YRS_NPA,215669,65.80
CUST_NO_OF_TIMES_NPA,215669,65.80
...,...,...
VINTAGE,2,0.00
KYC_FLG,2,0.00
INB_FLG,2,0.00
EKYC_FLG,2,0.00



4. TARGET ANALYSIS
Target missing values: 0

Target counts:


...


9. IDENTIFIER CHECK
UNIQUE_ID | Unique: 327741 | Missing: 0
TARGET | Unique: 2 | Missing: 0
TIME_PERIOD | Unique: 3 | Missing: 0

10. POTENTIAL OUTLIERS (IQR METHOD)


,Column,Lower_Bound,Upper_Bound,Outlier_Count,Outlier_Percentage
10,ONEMNTHOUTSTANGBAL,-2875.58,4951.48,52763,17.35
46,SEVENMNTHOUTSTANGBAL,-4577.16,7812.80,50551,17.00
34,FIVEMNTHOUTSTANGBAL,-3482.02,5973.73,50434,16.99
52,EIGHTMNTHOUTSTANGBAL,-5246.23,8935.05,50572,16.97
40,SIXMNTHOUTSTANGBAL,-3963.08,6782.24,50149,16.90
58,NINEMNTHOUTSTANGBAL,-5430.48,9246.06,50160,16.83
28,FOURMNTHOUTSTANGBAL,-3154.85,5423.93,50156,16.83
16,TWOMNTHOUTSTANGBAL,-2525.13,4369.53,50142,16.63
64,TENMNTHOUTSTANGBAL,-5384.54,9174.14,49463,16.61
22,THREEMNTHOUTSTANGBAL,-2661.50,4597.84,49667,16.58



11. NUMERICAL CORRELATION WITH TARGET


,Correlation_with_TARGET
CRIFF_11,-0.204853
TIMES_IRAC_SLIP,-0.181195
CRIFF_22,-0.162893
TIMES_IRAC_UPR,-0.161919
LOAN_TENURE,-0.140353
NO_YRS_RG3,-0.135916
FIRST_NPA_TENURE,-0.135651
LAST_1_YR_RG2,-0.134235
LATEST_NPA_TENURE,-0.129006
ACCT_AGE,-0.124073



EDA COMPLETE
